# Remove Background BYOC Example

This notebook demonstrates the BYOC package flow for **Remove Background**: request a short-lived CodeArtifact credential from the Bria Engine, install **`remove_bg`** from the **`bria-remove-bg`** repository, then run the local `RemoveBg` pipeline on a product image.

Remove Background segments the foreground subject out of an image and returns a cutout with a transparent alpha channel, powered by the **RMBG-2.0** model. It is the same component reused internally by [Packshot](../packshot/code_example.ipynb) and [Product Dimensions](../product_dimensions/code_example.ipynb), exposed here as a standalone pipeline.

**Prerequisites**

- `BRIA_API_TOKEN` in the environment.
- `HF_TOKEN` in the environment with access to `briaai/RMBG-2.0`.
- Python 3.10 or newer.
- Network access to Bria Engine, AWS CodeArtifact, and Hugging Face.
- An NVIDIA GPU runtime for the underlying model.

## 1. CodeArtifact Token

Request a short-lived credential for the **`bria-remove-bg`** PyPI repository. The credential is used as the CodeArtifact password with username `aws`.

In [ ]:
import json
import os
from urllib.error import HTTPError
from urllib.parse import urlencode
from urllib.request import Request, urlopen

BRIA_API_TOKEN = os.environ.get("BRIA_API_TOKEN")
if not BRIA_API_TOKEN:
    raise RuntimeError("Set BRIA_API_TOKEN in the environment before running this notebook.")

repository = "bria-remove-bg"
params = urlencode({"repository": repository})
url = f"https://engine.prod.bria-api.com/v2/auth/access/code_artifact?{params}"
request = Request(
    url,
    headers={
        "api_token": BRIA_API_TOKEN,
        "token": BRIA_API_TOKEN,
        "User-Agent": "BriaPlatform/APIdocs/LLMsAgent",
    },
)

try:
    with urlopen(request, timeout=60) as response:
        payload = json.loads(response.read().decode("utf-8"))
except HTTPError as exc:
    error_body = exc.read().decode("utf-8", errors="replace")
    try:
        error_payload = json.dumps(json.loads(error_body), indent=2)
    except json.JSONDecodeError:
        error_payload = error_body or "<empty response body>"
    raise RuntimeError(
        f"CodeArtifact token request failed for repository {repository!r} "
        f"with HTTP {exc.code}: {exc.reason}\n{error_payload}"
    ) from exc

result = payload.get("result") or {}
CODE_ARTIFACT_PASSWORD = (result.get("authorization_token") or "").strip()
if not CODE_ARTIFACT_PASSWORD:
    raise RuntimeError(f"CodeArtifact response did not include an authorization token: {payload}")

expiration = result.get("expiration")
print("CodeArtifact credential acquired." + (f" Expires: {expiration}" if expiration else ""))

## 2. Install dependencies

In [ ]:
import os
import subprocess
import sys
from pathlib import Path
from urllib.parse import quote

# Pip imports call os.getcwd(). If Jupyter started from a folder that was later removed, inherited cwd breaks pip.
try:
    pip_cwd = str(Path.cwd())
except FileNotFoundError:
    pip_cwd = str(Path.home())
    os.chdir(pip_cwd)

encoded_password = quote(CODE_ARTIFACT_PASSWORD, safe="")
remove_bg_index = (
    "https://aws:"
    + encoded_password
    + "@bria-300465780738.d.codeartifact.us-east-1.amazonaws.com/pypi/bria-remove-bg/simple/"
)

pytorch_index = "https://download.pytorch.org/whl/cu126"


def run_pip_install(install_cmd):
    redacted_cmd = " ".join(part.replace(encoded_password, "<redacted>") for part in install_cmd)
    print("Running:", redacted_cmd, flush=True)
    process = subprocess.Popen(
        install_cmd,
        cwd=pip_cwd,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        bufsize=1,
    )

    assert process.stdout is not None
    for line in process.stdout:
        print(line.replace(encoded_password, "<redacted>"), end="", flush=True)

    returncode = process.wait()
    if returncode != 0:
        raise RuntimeError("pip install failed. See pip output above; authenticated index URLs were redacted.")


run_pip_install(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--upgrade",
        "--force-reinstall",
        "torch",
        "torchvision",
        "--index-url",
        pytorch_index,
    ]
)

run_pip_install(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--upgrade",
        "--upgrade-strategy",
        "only-if-needed",
        "remove_bg",
        "--extra-index-url",
        remove_bg_index,
    ]
)

## 3. Start The Local Pipeline

The example disables model compilation for a simpler first run. Tune `RemoveBgConfig` for your deployment environment (e.g. set `compile_model=True` in a long-lived service for better steady-state throughput).

If the install cell changed core runtime packages, restart the notebook kernel before running this setup cell so Python imports the installed versions cleanly.

In [ ]:
import logging
from pathlib import Path

from remove_bg import RemoveBg, RemoveBgConfig

logging.basicConfig(level=logging.INFO)

outputs_dir = Path("outputs")
outputs_dir.mkdir(exist_ok=True)

remove_bg = RemoveBg(config=RemoveBgConfig(compile_model=False))
remove_bg.setup()
print("Remove Background pipeline setup complete.")

## 4. Remove The Background

Pass either a URL or a local image path into `RemoveBgInput.image`. The result is an RGBA image with the background made transparent.

In [ ]:
import time

from IPython.display import display
from remove_bg import RemoveBgInput

product_image = "https://bria-test-images.s3.us-east-1.amazonaws.com/strawberry.jpg"

task = RemoveBgInput(image=product_image)

t0 = time.time()
result = remove_bg.execute(task)
output_path = outputs_dir / "strawberry_cutout.png"
result.image.save(output_path)

print(f"Generated {result.image.size} cutout in {time.time() - t0:.1f}s")
print(f"Saved to {output_path}")
display(result.image)

## 5. Preview The Transparency

Alpha channels are hard to judge against a plain notebook background, so composite the cutout over a checkerboard pattern — the standard image-editor convention for "this area is transparent".

In [ ]:
import numpy as np
from PIL import Image

def checkerboard(size, tile=20, light=235, dark=200):
    w, h = size
    yy, xx = np.mgrid[0:h, 0:w]
    pattern = (((xx // tile) + (yy // tile)) % 2 == 0)
    board = np.where(pattern, light, dark).astype(np.uint8)
    return Image.fromarray(np.stack([board] * 3, axis=-1))

cutout = result.image.convert("RGBA")
board = checkerboard(cutout.size)
preview = Image.alpha_composite(board.convert("RGBA"), cutout).convert("RGB")

preview_path = outputs_dir / "strawberry_cutout_preview.png"
preview.save(preview_path)
print(f"Saved transparency preview to {preview_path}")
display(preview)

## 6. Cleanup

Release resources held by the underlying pipeline when you are done.

In [ ]:
remove_bg.cleanup()
print("Cleanup done.")